# Phase 3: Fine-Tuning XLM-RoBERTa for Multilingual Misinformation Detection

This notebook is designed to run on Google Colab using a free T4 GPU. It fine-tunes xlm-roberta-base on the unified master_dataset.csv and pushes the weights to the Hugging Face Hub.

In [ ]:
!pip install transformers==4.46.3 datasets torch scikit-learn sentencepiece huggingface_hub pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Instructions for the user:
# 1. Upload `master_dataset.csv` to the root of your Google Drive.
# 2. Or upload it directly to the Colab environment.
import os
from pathlib import Path

drive_path = '/content/drive/MyDrive/master_dataset.csv'
local_path = 'master_dataset.csv'

if os.path.exists(drive_path):
    data_path = drive_path
    print("Found dataset in Google Drive.")
elif os.path.exists(local_path):
    data_path = local_path
    print("Found dataset in local Colab environment.")
else:
    from google.colab import files
    print("Please upload `master_dataset.csv`.")
    uploaded = files.upload()
    data_path = local_path

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(data_path)
# Drop any potential NaNs just in case
df = df.dropna(subset=['text', 'label'])

# Stratified split: 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['language'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['language'], random_state=42)

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

In [ ]:
from transformers import AutoTokenizer

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_length = 256

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)

from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

In [ ]:
from transformers import AutoModelForSequenceClassification

# 0 = Real, 1 = Fake
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/misinfo-xlmr-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True, # Enable mixed precision for T4 GPU
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    return {"accuracy": acc, "f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

Start Training
This will take approximately 1-2 hours on a Colab T4 GPU.

In [ ]:
trainer.train()

In [ ]:
test_dataset = Dataset.from_pandas(test_df)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Evaluate overall metrics
metrics = trainer.evaluate(test_dataset)
print("Overall Test Metrics:", metrics)

# Evaluate per-language F1
import torch
from torch.utils.data import DataLoader

# Ensure model is on GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

test_preds = []
test_labels = []

model.eval()
with torch.no_grad():
    for batch in DataLoader(test_dataset, batch_size=16):
        inputs = {k: v.to(device) for k, v in batch.items() if k in ['input_ids', 'attention_mask']}
        labels = batch['label'].to(device)
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=-1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

test_df['preds'] = test_preds

print("\nPer-Language F1 Scores:")
for lang in test_df['language'].unique():
    subset = test_df[test_df['language'] == lang]
    f1 = f1_score(subset['label'], subset['preds'], average='binary')
    print(f"{lang}: {f1:.4f}")

Push to Hugging Face Hub
Enter your Hugging Face token when prompted. The model will be pushed to hf.co/<your-username>/misinfo-xlmr-nli.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# Push the model and tokenizer
model.push_to_hub("misinfo-xlmr-nli", private=True)
tokenizer.push_to_hub("misinfo-xlmr-nli", private=True)
print("Model successfully pushed to Hugging Face Hub.")